In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

SORTIE = RACINE / "data" / "processed"

valeurs = pd.read_csv(SORTIE / "valeurs_portefeuilles.csv", index_col=0)
comparaisons = pd.read_csv(SORTIE / "valeurs_comparaisons.csv", index_col=0)
tout = valeurs.join(comparaisons)

print(valeurs.shape, comparaisons.shape)

(6709, 20) (6709, 5)


In [2]:
lignes = []
for serie in tout.columns:
    niveau = tout[serie].dropna()
    quotidien = niveau.pct_change().dropna()

    mensuel = niveau.groupby(pd.Series(niveau.index).str[:7].values).last()
    mensuel = mensuel.pct_change().dropna()

    lignes.append({"serie": serie,
                   "vol_quotidienne": quotidien.std() * np.sqrt(252),
                   "vol_mensuelle": mensuel.std() * np.sqrt(12),
                   "autocorrelation": quotidien.autocorr(1),
                   "obs_quotidiennes": len(quotidien),
                   "obs_mensuelles": len(mensuel)})

frequence = pd.DataFrame(lignes).set_index("serie")
frequence["rapport"] = frequence.vol_quotidienne / frequence.vol_mensuelle

print(frequence.round(4).to_string())
print()
print("rapport median %.3f | min %.3f | max %.3f"
      % (frequence.rapport.median(), frequence.rapport.min(), frequence.rapport.max()))

          vol_quotidienne  vol_mensuelle  autocorrelation  obs_quotidiennes  obs_mensuelles  rapport
serie                                                                                               
P1_reeq            0.2173         0.1910          -0.0446              6708             320   1.1375
P1_cons            0.2352         0.1997          -0.0569              6708             320   1.1777
P10_reeq           0.2302         0.1986          -0.0251              6708             320   1.1590
P10_cons           0.2722         0.2592          -0.0319              6708             320   1.0504
P2_reeq            0.2382         0.2146          -0.0298              6708             320   1.1097
P2_cons            0.2556         0.2212          -0.0446              6708             320   1.1552
P3_reeq            0.1846         0.1517          -0.0785              6708             320   1.2166
P3_cons            0.2300         0.2164          -0.0547              6708             320

In [3]:
FENETRES = [252, 756]

lignes = []
for serie in tout.columns:
    r = tout[serie].dropna().pct_change().dropna()
    ligne = {"serie": serie, "periode_complete": r.std() * np.sqrt(252)}
    for n in FENETRES:
        glissante = r.rolling(n).std() * np.sqrt(252)
        ligne[f"med_{n}"] = glissante.median()
        ligne[f"min_{n}"] = glissante.min()
        ligne[f"max_{n}"] = glissante.max()
        ligne[f"amplitude_{n}"] = glissante.max() / glissante.min()
    lignes.append(ligne)

fenetres = pd.DataFrame(lignes).set_index("serie")
print(fenetres[["periode_complete", "med_252", "min_252", "max_252", "amplitude_252"]]
      .round(4).to_string())
print()
print("erreur relative d'estimation : 252 jours %.2f %%, 756 jours %.2f %%"
      % (100 / np.sqrt(2 * 252), 100 / np.sqrt(2 * 756)))

          periode_complete  med_252  min_252  max_252  amplitude_252
serie                                                               
P1_reeq             0.2173   0.1753   0.0891   0.4737         5.3154
P1_cons             0.2352   0.1999   0.0961   0.4861         5.0606
P10_reeq            0.2302   0.1863   0.1078   0.5658         5.2489
P10_cons            0.2722   0.2093   0.1120   0.6069         5.4185
P2_reeq             0.2382   0.1944   0.1006   0.4949         4.9174
P2_cons             0.2556   0.2193   0.1082   0.4999         4.6198
P3_reeq             0.1846   0.1497   0.0787   0.4271         5.4272
P3_cons             0.2300   0.1813   0.0918   0.5026         5.4749
P4_reeq             0.3115   0.2309   0.1183   0.5973         5.0480
P4_cons             0.3436   0.2588   0.1459   0.6458         4.4255
P5_reeq             0.3218   0.2574   0.1444   0.6342         4.3916
P5_cons             0.3473   0.2826   0.1471   0.6311         4.2897
P6_reeq             0.1894   0.161

In [4]:
SEUIL_TENSION = 0.15
SEUIL_MAJEUR = 0.20
REFERENCE = "SPY"


def episodes_de_tension(niveau, seuil):
    sommet = niveau.cummax()
    groupe = (sommet != sommet.shift()).cumsum()
    lignes = []
    for _, segment in niveau.groupby(groupe):
        repli = segment / segment.iloc[0] - 1
        if repli.min() <= -seuil:
            dernier = segment.index[-1]
            retour = (niveau.index[niveau.index.get_loc(dernier) + 1]
                      if dernier != niveau.index[-1] else None)
            lignes.append({"debut": segment.index[0], "creux": repli.idxmin(),
                           "repli": repli.min(), "retour": retour,
                           "seances": len(segment)})
    return pd.DataFrame(lignes)


marche = tout[REFERENCE].dropna()
tensions = episodes_de_tension(marche, SEUIL_TENSION)
tensions["majeur"] = tensions.repli <= -SEUIL_MAJEUR

en_tension = pd.Series(False, index=tout.index)
for r in tensions.itertuples():
    en_tension.loc[r.debut:r.creux] = True

tensions.to_csv(SORTIE / "episodes_tension.csv", index=False, encoding="utf-8")

print(tensions.to_string(index=False, float_format=lambda x: f"{x:.1%}"))
print()
print("seances en tension : %d sur %d, soit %.1f %%"
      % (en_tension.sum(), len(en_tension), 100 * en_tension.mean()))
print("episodes majeurs :", int(tensions.majeur.sum()))

     debut      creux  repli     retour  seances  majeur
2000-03-24 2002-10-09 -47.3% 2006-11-14     1670    True
2007-10-09 2009-03-09 -54.9% 2012-08-16     1224    True
2018-09-20 2018-12-24 -19.1% 2019-04-08      136   False
2020-02-19 2020-03-23 -33.7% 2020-08-10      120    True
2022-01-03 2022-10-12 -24.4% 2023-12-13      489    True
2025-02-19 2025-04-08 -18.7% 2025-06-26       88   False

seances en tension : 1315 sur 6709, soit 19.6 %
episodes majeurs : 4


In [5]:
import yfinance as yf

TAUX = RACINE / "data" / "raw" / "taux_sans_risque.csv"

if TAUX.exists():
    print(f"{TAUX.name} existe deja, collecte ignoree")
else:
    h = yf.Ticker("^IRX").history(period="max", auto_adjust=False)
    if h.empty:
        raise ValueError("aucune donnee pour ^IRX")
    serie = h["Close"].rename("taux_annuel_pct")
    serie.index = [str(d.date()) for d in serie.index]
    serie.index.name = "date"
    serie.to_frame().assign(symbole="^IRX").to_csv(TAUX, encoding="utf-8")
    print(len(serie), "lignes ecrites dans", TAUX.name)

taux = pd.read_csv(TAUX, index_col=0)["taux_annuel_pct"]
taux = taux.reindex(tout.index).ffill()

sans_risque = (1 + taux / 100) ** (1 / 252) - 1

print("couverture :", int(taux.notna().sum()), "sur", len(tout.index))
print("taux annuel : moyenne %.2f %% | min %.2f %% | max %.2f %%"
      % (taux.mean(), taux.min(), taux.max()))
print("jours a taux negatif :", int((taux < 0).sum()))

16655 lignes ecrites dans taux_sans_risque.csv
couverture : 6709 sur 6709
taux annuel : moyenne 1.92 % | min -0.10 % | max 6.22 %
jours a taux negatif : 7


In [6]:
lignes = []
for serie in tout.columns:
    niveau = tout[serie].dropna()
    quotidien = niveau.pct_change().dropna()
    mensuel = niveau.groupby(pd.Series(niveau.index).str[:7].values).last().pct_change().dropna()

    lignes.append({
        "serie": serie,
        "vol_quotidienne": quotidien.std() * np.sqrt(252),
        "med_glissante_252": (quotidien.rolling(252).std() * np.sqrt(252)).median(),
        "vol_mensuelle": mensuel.std() * np.sqrt(12),
        "rendement_annualise": (niveau.iloc[-1] / niveau.iloc[0]) ** (252 / len(niveau)) - 1,
    })

volatilite = pd.DataFrame(lignes).set_index("serie").sort_values("vol_quotidienne")
volatilite["contre_SPY"] = volatilite.vol_quotidienne / volatilite.loc["SPY", "vol_quotidienne"]

volatilite.to_csv(SORTIE / "risque_volatilite.csv", encoding="utf-8")
print(volatilite.round(4).to_string())

          vol_quotidienne  med_glissante_252  vol_mensuelle  rendement_annualise  contre_SPY
serie                                                                                       
P3_reeq            0.1846             0.1497         0.1517               0.1426      0.9662
P6_reeq            0.1894             0.1611         0.1572               0.1191      0.9911
SPY                0.1911             0.1534         0.1501               0.0827      1.0000
GSPC               0.1924             0.1556         0.1512               0.0647      1.0067
SP500TR            0.1924             0.1556         0.1513               0.0844      1.0067
P6_cons            0.1953             0.1619         0.1603               0.1167      1.0223
RSP                0.1958             0.1465         0.1627               0.1127      1.0247
SPXEW              0.2094             0.1608         0.1735               0.0816      1.0960
P1_reeq            0.2173             0.1753         0.1910           

In [7]:
lignes = []
for serie in tout.columns:
    r = tout[serie].dropna().pct_change().dropna()
    n = len(r)
    asymetrie, aplatissement = r.skew(), r.kurt()
    centre = (r - r.mean()) / r.std()

    lignes.append({
        "serie": serie,
        "asymetrie": asymetrie,
        "aplatissement_exces": aplatissement,
        "jarque_bera": n / 6 * (asymetrie ** 2 + aplatissement ** 2 / 4),
        "au_dela_3sigma_pct": 100 * (centre.abs() > 3).mean(),
        "var99_historique": -r.quantile(0.01),
        "var99_gaussienne": -(r.mean() - 2.326 * r.std()),
    })

forme = pd.DataFrame(lignes).set_index("serie")
forme["erreur_var99_pct"] = 100 * (forme.var99_gaussienne / forme.var99_historique - 1)
forme.to_csv(SORTIE / "risque_distribution.csv", encoding="utf-8")

print(forme.round(3).to_string())
print()
print("sous la loi normale : asymetrie 0, aplatissement excedentaire 0, au-dela de 3 sigma 0,27 %")
print("seuil de Jarque-Bera a 5 %% : 5,99 | series le depassant : %d sur %d"
      % ((forme.jarque_bera > 5.99).sum(), len(forme)))

          asymetrie  aplatissement_exces  jarque_bera  au_dela_3sigma_pct  var99_historique  var99_gaussienne  erreur_var99_pct
serie                                                                                                                          
P1_reeq      -0.148                7.083    14047.673               1.357             0.036             0.031           -13.300
P1_cons      -0.180                7.751    16827.601               1.521             0.042             0.034           -20.553
P10_reeq     -0.217               11.712    38390.668               1.401             0.038             0.033           -13.141
P10_cons     -0.090               11.720    38402.715               1.565             0.048             0.039           -18.188
P2_reeq      -0.079                5.489     8426.637               1.416             0.040             0.034           -14.087
P2_cons      -0.133                5.868     9643.760               1.550             0.045             

In [8]:
lignes = []
for serie in tout.columns:
    niveau = tout[serie].dropna()
    repli = niveau / niveau.cummax() - 1

    creux = repli.idxmin()
    sommet = niveau.loc[:creux].idxmax()
    apres = niveau.loc[creux:]
    rattrape = apres[apres >= niveau.loc[sommet]]
    retour = rattrape.index[0] if len(rattrape) else None

    position = niveau.index.get_loc
    lignes.append({
        "serie": serie,
        "repli_max": repli.min(),
        "sommet": sommet, "creux": creux, "retour": retour,
        "chute_seances": position(creux) - position(sommet),
        "recuperation_seances": position(retour) - position(creux) if retour else None,
        "part_sous_l_eau": 100 * (repli < -0.0001).mean(),
    })

replis = pd.DataFrame(lignes).set_index("serie").sort_values("repli_max")
replis.to_csv(SORTIE / "risque_replis.csv", encoding="utf-8")
print(replis.to_string(float_format=lambda x: f"{x:.2f}"))

          repli_max      sommet       creux      retour  chute_seances  recuperation_seances  part_sous_l_eau
serie                                                                                                        
P4_cons       -0.87  2000-03-27  2002-10-07  2012-03-08            634                  2372            94.13
P9_cons       -0.73  2000-03-27  2002-10-09  2007-10-17            636                  1264            93.96
P5_cons       -0.73  2000-09-01  2002-10-09  2007-06-15            525                  1178            91.56
P5_reeq       -0.73  2000-09-01  2002-10-09  2005-12-02            525                   794            90.58
P9_reeq       -0.72  2000-03-27  2002-10-09  2007-10-17            636                  1264            92.58
P4_reeq       -0.71  2000-03-27  2002-07-25  2003-07-08            583                   239            89.45
P8_reeq       -0.65  2000-03-22  2002-10-09  2003-07-24            639                   198            90.55
P10_cons  

In [9]:
taux = pd.read_csv(RACINE / "data" / "raw" / "taux_sans_risque.csv",
                   index_col=0)["taux_annuel_pct"].reindex(tout.index).ffill()
sans_risque = (1 + taux / 100) ** (1 / 252) - 1

lignes = []
for serie in tout.columns:
    r = tout[serie].dropna().pct_change().dropna()
    excedent = (r - sans_risque.reindex(r.index)).dropna()

    volatilite = r.std() * np.sqrt(252)
    baisse = np.sqrt((np.minimum(excedent, 0) ** 2).mean()) * np.sqrt(252)

    lignes.append({"serie": serie,
                   "volatilite": volatilite,
                   "volatilite_baisse": baisse,
                   "sharpe": excedent.mean() * np.sqrt(252) / r.std(),
                   "sortino": excedent.mean() * 252 / baisse})

ratios = pd.DataFrame(lignes).set_index("serie").sort_values("sharpe", ascending=False)
ratios.to_csv(SORTIE / "risque_ratios.csv", encoding="utf-8")
print(ratios.round(3).to_string())

          volatilite  volatilite_baisse  sharpe  sortino
serie                                                   
P4_reeq        0.312              0.210   0.840    1.244
P1_reeq        0.217              0.152   0.811    1.163
P2_reeq        0.238              0.165   0.782    1.125
P1_cons        0.235              0.166   0.770    1.094
P2_cons        0.256              0.179   0.744    1.063
P7_reeq        0.231              0.161   0.741    1.063
P5_cons        0.347              0.238   0.719    1.052
P7_cons        0.239              0.167   0.719    1.028
P3_reeq        0.185              0.130   0.713    1.011
P5_reeq        0.322              0.221   0.691    1.007
P8_reeq        0.266              0.182   0.656    0.961
P3_cons        0.230              0.161   0.633    0.903
P10_reeq       0.230              0.163   0.608    0.860
P10_cons       0.272              0.191   0.606    0.864
P6_reeq        0.189              0.134   0.590    0.832
P6_cons        0.195           

In [10]:
debut_commun = tout["RSP"].dropna().index[0]
fenetre = tout.loc[debut_commun:]

lignes = []
for serie in fenetre.columns:
    niveau = fenetre[serie].dropna()
    if len(niveau) < 0.9 * len(fenetre):
        continue
    r = niveau.pct_change().dropna()
    excedent = (r - sans_risque.reindex(r.index)).dropna()
    baisse = np.sqrt((np.minimum(excedent, 0) ** 2).mean()) * np.sqrt(252)

    lignes.append({"serie": serie,
                   "annualise": (niveau.iloc[-1] / niveau.iloc[0]) ** (252 / len(niveau)) - 1,
                   "volatilite": r.std() * np.sqrt(252),
                   "sharpe": excedent.mean() * np.sqrt(252) / r.std(),
                   "sortino": excedent.mean() * 252 / baisse,
                   "repli_max": (niveau / niveau.cummax() - 1).min()})

commun = pd.DataFrame(lignes).set_index("serie").sort_values("sharpe", ascending=False)
commun.to_csv(SORTIE / "risque_fenetre_commune.csv", encoding="utf-8")
print("fenetre commune :", debut_commun, "->", fenetre.index[-1])
print(commun.round(3).to_string())

fenetre commune : 2003-05-01 -> 2026-09-04
          annualise  volatilite  sharpe  sortino  repli_max
serie                                                      
P4_reeq       0.312       0.271   1.076    1.598     -0.576
P1_reeq       0.214       0.210   0.947    1.351     -0.518
P2_reeq       0.225       0.226   0.937    1.339     -0.555
P5_cons       0.284       0.317   0.894    1.295     -0.603
P1_cons       0.218       0.235   0.887    1.257     -0.533
P5_reeq       0.253       0.283   0.880    1.266     -0.613
P2_cons       0.229       0.251   0.880    1.250     -0.555
P4_cons       0.260       0.311   0.845    1.225     -0.650
P8_reeq       0.216       0.257   0.823    1.207     -0.583
P7_reeq       0.196       0.229   0.821    1.173     -0.540
P7_cons       0.193       0.239   0.787    1.120     -0.537
P3_reeq       0.151       0.182   0.771    1.097     -0.459
P9_reeq       0.211       0.291   0.747    1.080     -0.634
P10_cons      0.182       0.282   0.675    0.962     -0.6

In [11]:
rendements = pd.read_csv(SORTIE / "rendements_valorisation.csv", index_col=0)
appartenance = pd.read_csv(SORTIE / "appartenance.csv")


def correlation_moyenne(colonnes, periode=None):
    bloc = rendements[colonnes] if periode is None else rendements.loc[periode, colonnes]
    matrice = bloc.corr(min_periods=250).to_numpy()
    haut = np.triu_indices_from(matrice, 1)
    paires = matrice[haut]
    paires = paires[~np.isnan(paires)]
    return paires.mean(), len(paires)


lignes = []
for pf, groupe in appartenance.groupby("portefeuille"):
    colonnes = [c for c in groupe.titre if c in rendements.columns]
    rho, paires = correlation_moyenne(colonnes)
    lignes.append({"portefeuille": pf, "titres": len(colonnes),
                   "rho_moyen": rho, "paires": paires})

correlations = pd.DataFrame(lignes).set_index("portefeuille").reindex(
    [f"P{i}" for i in range(1, 11)])
correlations.to_csv(SORTIE / "risque_correlations.csv", encoding="utf-8")
print(correlations.round(4).to_string())

              titres  rho_moyen  paires
portefeuille                           
P1               135     0.3380    8911
P2               111     0.3581    5995
P3                24     0.4107     276
P4                11     0.3833      55
P5                34     0.4177     561
P6                24     0.5266     276
P7                28     0.4774     378
P8                 7     0.4552      21
P9                13     0.4740      66
P10               18     0.4115     153


In [12]:
colonnes = [c for c in appartenance[appartenance.portefeuille == "P1"].titre
            if c in rendements.columns]
for a, b in [("2000-01-03", "2007-12-31"), ("2008-01-01", "2015-12-31"),
             ("2016-01-01", "2026-09-04")]:
    rho, n = correlation_moyenne(colonnes, slice(a, b))
    print(f"P1 {a[:4]}-{b[:4]} : rho {rho:.4f} sur {n} paires")

P1 2000-2007 : rho 0.2290 sur 5356 paires
P1 2008-2015 : rho 0.4347 sur 7140 paires
P1 2016-2026 : rho 0.3524 sur 8911 paires


In [13]:
def rho_implicite(bloc):
    ecarts = bloc.std().dropna()
    n = len(ecarts)
    if n < 10:
        return np.nan
    variance_portefeuille = bloc[ecarts.index].mean(axis=1).var()
    somme_carres = (ecarts ** 2).sum()
    return (n ** 2 * variance_portefeuille - somme_carres) / (ecarts.sum() ** 2 - somme_carres)


colonnes = [c for c in appartenance[appartenance.portefeuille == "P1"].titre
            if c in rendements.columns]
bloc = rendements[colonnes]

print("rho implicite sur toute la periode : %.4f" % rho_implicite(bloc))
print("rho direct par paires              : %.4f" % correlation_moyenne(colonnes)[0])

glissante = []
for i in range(252, len(bloc) + 1, 21):
    fenetre = bloc.iloc[i - 252:i]
    glissante.append({"date": bloc.index[i - 1], "rho": rho_implicite(fenetre)})

glissante = pd.DataFrame(glissante).set_index("date")
glissante.to_csv(SORTIE / "risque_correlation_glissante.csv", encoding="utf-8")

print("min %.3f (%s) | mediane %.3f | max %.3f (%s)"
      % (glissante.rho.min(), glissante.rho.idxmin(), glissante.rho.median(),
         glissante.rho.max(), glissante.rho.idxmax()))
print(glissante.rho.groupby(pd.Series(glissante.index).str[:4].values).mean().round(3).to_string())

rho implicite sur toute la periode : 0.3133
rho direct par paires              : 0.3380
min 0.154 (2000-12-29) | mediane 0.299 | max 0.579 (2009-09-09)
2000    0.154
2001    0.210
2002    0.250
2003    0.292
2004    0.262
2005    0.244
2006    0.246
2007    0.273
2008    0.377
2009    0.551
2010    0.468
2011    0.459
2012    0.511
2013    0.313
2014    0.268
2015    0.308
2016    0.331
2017    0.219
2018    0.250
2019    0.336
2020    0.517
2021    0.341
2022    0.372
2023    0.423
2024    0.251
2025    0.338
2026    0.266


In [14]:
lignes = []
for pf, groupe in appartenance.groupby("portefeuille"):
    colonnes = [c for c in groupe.titre if c in rendements.columns]
    bloc = rendements[colonnes]

    ecarts = (bloc.std() * np.sqrt(252)).dropna()
    n = len(ecarts)
    sigma = ecarts.mean()
    rho = correlation_moyenne(list(ecarts.index))[0]

    predite = sigma * np.sqrt(rho + (1 - rho) / n)
    plancher = sigma * np.sqrt(rho)
    reelle = valeurs[f"{pf}_reeq"].pct_change().std() * np.sqrt(252)

    lignes.append({"portefeuille": pf, "n": n, "sigma_moyen": sigma, "rho": rho,
                   "vol_predite": predite, "vol_reelle": reelle,
                   "plancher_n_infini": plancher,
                   "n_effectif": 1 / (rho + (1 - rho) / n),
                   "reduction_obtenue": 1 - predite / sigma,
                   "reste_a_gagner": (predite - plancher) / sigma})

decomposition = pd.DataFrame(lignes).set_index("portefeuille").reindex(
    [f"P{i}" for i in range(1, 11)])
decomposition.to_csv(SORTIE / "risque_decomposition.csv", encoding="utf-8")
print(decomposition.round(4).to_string())

                n  sigma_moyen     rho  vol_predite  vol_reelle  plancher_n_infini  n_effectif  reduction_obtenue  reste_a_gagner
portefeuille                                                                                                                     
P1            135       0.3903  0.3380       0.2285      0.2173             0.2269      2.9166             0.4145          0.0042
P2            111       0.4083  0.3581       0.2463      0.2382             0.2443      2.7478             0.3967          0.0048
P3             24       0.3072  0.4107       0.2026      0.1846             0.1968      2.2975             0.3403          0.0189
P4             11       0.4425  0.3833       0.2933      0.3115             0.2739      2.2760             0.3371          0.0437
P5             34       0.4877  0.4177       0.3216      0.3218             0.3152      2.3000             0.3406          0.0131
P6             24       0.2765  0.5266       0.2044      0.1894             0.2006      1.

In [20]:
poids_mensuels = pd.read_csv(SORTIE / "poids_mensuels.csv")
derniere = poids_mensuels.date.max()

ordre = sorted(poids_mensuels.serie.unique(),
               key=lambda s: (int(s.split("_")[0][1:]), s))


def contributions(serie, fenetre=252):
    w = poids_mensuels[(poids_mensuels.serie == serie)
                       & (poids_mensuels.date == derniere)
                       & (poids_mensuels.titre != "_tresorerie")].set_index("titre").poids
    w = w[w > 0]
    colonnes = [c for c in w.index if c in rendements.columns]
    w = w[colonnes] / w[colonnes].sum()

    covariance = rendements[colonnes].iloc[-fenetre:].cov() * 252
    sigma_w = covariance.to_numpy() @ w.to_numpy()
    volatilite = np.sqrt(float(w.to_numpy() @ sigma_w))
    contribution = w.to_numpy() * sigma_w / volatilite

    table = pd.DataFrame({"poids": w.to_numpy(), "contribution": contribution,
                          "part_du_risque": contribution / volatilite}, index=colonnes)
    return table.sort_values("part_du_risque", ascending=False), volatilite


detail, resume = [], []

for serie in ordre:
    table, volatilite = contributions(serie)

    bloc = table.reset_index().rename(columns={"index": "titre"})
    bloc.insert(0, "serie", serie)
    bloc["rang"] = range(1, len(bloc) + 1)
    detail.append(bloc)

    resume.append({"serie": serie, "titres": len(table), "vol": volatilite,
                   "somme": table.contribution.sum(),
                   "premier": table.index[0],
                   "poids_1": table.poids.iloc[0],
                   "risque_1": table.part_du_risque.iloc[0],
                   "poids_5": table.poids.head(5).sum(),
                   "risque_5": table.part_du_risque.head(5).sum()})

    print(f"\n=== {serie} | {len(table)} titres | volatilite {volatilite:.2%} "
          f"| somme des contributions {table.contribution.sum():.2%}")
    print(table.head(10).to_string(float_format=lambda x: f"{x:.4f}"))
    print("les 5 premiers : %.1f %% du poids, %.1f %% du risque"
          % (100 * table.poids.head(5).sum(), 100 * table.part_du_risque.head(5).sum()))

detail = pd.concat(detail, ignore_index=True)
detail.to_csv(SORTIE / "risque_contributions_detail.csv", index=False, encoding="utf-8")

resume = pd.DataFrame(resume).set_index("serie")
resume["amplif"] = resume.risque_5 / resume.poids_5
resume.to_csv(SORTIE / "risque_contributions.csv", encoding="utf-8")

print("\n\n=== synthese des vingt series ===")
print("identite verifiee :", bool(np.allclose(resume.vol, resume.somme)))
print(resume.drop(columns="somme").round(4).to_string())


=== P1_cons | 135 titres | volatilite 40.99% | somme des contributions 40.99%
      poids  contribution  part_du_risque
SNDK 0.1938        0.2037          0.4969
NVDA 0.1944        0.0462          0.1128
AVGO 0.0516        0.0155          0.0378
LITE 0.0228        0.0148          0.0361
VRT  0.0218        0.0102          0.0249
WDC  0.0149        0.0093          0.0226
FIX  0.0212        0.0092          0.0225
TSLA 0.0413        0.0088          0.0216
ANET 0.0232        0.0071          0.0172
GEV  0.0208        0.0068          0.0166
les 5 premiers : 48.4 % du poids, 70.9 % du risque

=== P1_reeq | 135 titres | volatilite 24.93% | somme des contributions 24.93%
      poids  contribution  part_du_risque
SNDK 0.0368        0.0328          0.1314
MU   0.0188        0.0118          0.0475
STX  0.0172        0.0090          0.0362
WDC  0.0145        0.0088          0.0353
DELL 0.0239        0.0087          0.0348
LITE 0.0133        0.0086          0.0345
MRVL 0.0146        0.0075          

In [21]:
def concentration(serie, date=None, fenetre=252):
    date = date or derniere
    w = poids_mensuels[(poids_mensuels.serie == serie)
                       & (poids_mensuels.date == date)
                       & (poids_mensuels.titre != "_tresorerie")].set_index("titre").poids
    w = w[w > 0]
    colonnes = [c for c in w.index if c in rendements.columns]
    w = w[colonnes] / w[colonnes].sum()

    fin = rendements.index.get_loc(date) + 1
    covariance = rendements[colonnes].iloc[max(0, fin - fenetre):fin].cov() * 252
    sigma_w = covariance.to_numpy() @ w.to_numpy()
    variance = float(w.to_numpy() @ sigma_w)
    part_risque = w.to_numpy() * sigma_w / variance

    h_poids = float((w.to_numpy() ** 2).sum())
    h_risque = float((part_risque ** 2).sum())

    return {"serie": serie, "titres": len(colonnes),
            "hhi_poids": h_poids, "n_eff_poids": 1 / h_poids,
            "hhi_risque": h_risque, "n_eff_risque": 1 / h_risque,
            "rapport": h_risque / h_poids}


concentrations = pd.DataFrame([concentration(s) for s in ordre]).set_index("serie")
concentrations.to_csv(SORTIE / "risque_concentration.csv", encoding="utf-8")

print("au", derniere)
print(concentrations.round(4).to_string())

au 2026-09-04
          titres  hhi_poids  n_eff_poids  hhi_risque  n_eff_risque  rapport
serie                                                                      
P1_cons      135     0.0864      11.5687      0.2665        3.7516   3.0836
P1_reeq      135     0.0094     106.8626      0.0337       29.6583   3.6031
P2_cons      111     0.1012       9.8776      0.2853        3.5055   2.8178
P2_reeq      111     0.0116      86.1904      0.0358       27.9269   3.0863
P3_cons       24     0.3344       2.9904      0.7937        1.2599   2.3734
P3_reeq       24     0.0423      23.6670      0.0451       22.1901   1.0666
P4_cons       11     0.2384       4.1939      0.3568        2.8024   1.4965
P4_reeq       11     0.0967      10.3391      0.1090        9.1764   1.1267
P5_cons       34     0.2787       3.5877      0.5760        1.7360   2.0667
P5_reeq       34     0.0402      24.8694      0.0816       12.2609   2.0284
P6_cons       24     0.0768      13.0133      0.2018        4.9565   2.625

In [24]:
NOMS = {"P4": "acheteurs", "P5": "vendeurs", "P6": "electricite",
        "P7": "equipement", "P8": "immobilier", "P9": "amont des puces",
        "P10": "matieres et energie"}

maillon = {}
for pf, nom in NOMS.items():
    for titre in appartenance[appartenance.portefeuille == pf].titre:
        maillon[titre] = nom


def par_maillon(serie, fenetre=252):
    w = poids_mensuels[(poids_mensuels.serie == serie)
                       & (poids_mensuels.date == derniere)
                       & (poids_mensuels.titre != "_tresorerie")].set_index("titre").poids
    w = w[w > 0]
    colonnes = [c for c in w.index if c in rendements.columns]
    w = w[colonnes] / w[colonnes].sum()

    covariance = rendements[colonnes].iloc[-fenetre:].cov() * 252
    sigma_w = covariance.to_numpy() @ w.to_numpy()
    variance = float(w.to_numpy() @ sigma_w)
    part = w.to_numpy() * sigma_w / variance

    d = pd.DataFrame({"titre": colonnes, "poids": w.to_numpy(), "part": part})
    d["maillon"] = d.titre.map(maillon)

    groupe = d.groupby("maillon").agg(titres=("titre", "size"),
                                      poids=("poids", "sum"),
                                      risque=("part", "sum"))
    groupe["amplification"] = groupe.risque / groupe.poids
    return groupe.sort_values("risque", ascending=False), np.sqrt(variance)


tableaux = []
for serie in ordre:
    groupe, volatilite = par_maillon(serie)
    print(f"=== {serie} | vol {volatilite:.2%} | {len(groupe)} maillon(s) "
          f"| somme {groupe.risque.sum():.4f}")
    print(groupe.round(4).to_string())
    print()
    bloc = groupe.reset_index()
    bloc.insert(0, "serie", serie)
    tableaux.append(bloc)

pd.concat(tableaux, ignore_index=True).to_csv(
    SORTIE / "risque_par_maillon.csv", index=False, encoding="utf-8")

=== P1_cons | vol 40.99% | 7 maillon(s) | somme 1.0000
                     titres   poids  risque  amplification
maillon                                                   
vendeurs                 34  0.6052  0.8143         1.3454
equipement               28  0.1543  0.1019         0.6601
acheteurs                11  0.0828  0.0296         0.3573
amont des puces          13  0.0345  0.0288         0.8362
matieres et energie      18  0.0672  0.0121         0.1803
electricite              24  0.0432  0.0120         0.2766
immobilier                7  0.0128  0.0014         0.1086

=== P1_reeq | vol 24.93% | 7 maillon(s) | somme 1.0000
                     titres   poids  risque  amplification
maillon                                                   
vendeurs                 34  0.3520  0.6182         1.7564
equipement               28  0.1901  0.1575         0.8284
amont des puces          13  0.0984  0.1480         1.5032
matieres et energie      18  0.1186  0.0292         0.2464
elec

In [25]:
episodes = pd.read_csv(SORTIE / "episodes_tension.csv")

lignes = []
for e in episodes.itertuples():
    for serie in tout.columns:
        s = tout[serie].loc[e.debut:e.creux].dropna()
        if len(s) < 20:
            lignes.append({"episode": e.debut, "serie": serie,
                           "rendement": np.nan, "vol": np.nan, "seances": len(s)})
            continue
        r = s.pct_change().dropna()
        lignes.append({"episode": e.debut, "serie": serie,
                       "rendement": s.iloc[-1] / s.iloc[0] - 1,
                       "vol": r.std() * np.sqrt(252), "seances": len(s)})

crises = pd.DataFrame(lignes)
crises.to_csv(SORTIE / "risque_par_episode.csv", index=False, encoding="utf-8")

rang = [f"P{i}_{v}" for i in range(1, 11) for v in ("reeq", "cons")] \
       + ["SPY", "RSP", "GSPC", "SP500TR", "SPXEW"]

for grandeur, titre in [("rendement", "RENDEMENT PAR EPISODE, du pic au creux"),
                        ("vol", "VOLATILITE ANNUALISEE PAR EPISODE")]:
    table = crises.pivot(index="serie", columns="episode", values=grandeur)
    table = table.reindex([x for x in rang if x in table.index])
    print(titre)
    print((table * 100).round(1).to_string())
    print()

RENDEMENT PAR EPISODE, du pic au creux
episode   2000-03-24  2007-10-09  2018-09-20  2020-02-19  2022-01-03  2025-02-19
serie                                                                           
P1_reeq        -37.6       -50.0       -18.5       -36.8       -19.7       -21.4
P1_cons        -36.6       -50.4       -22.3       -37.8       -28.0       -28.1
P2_reeq        -46.5       -53.6       -20.5       -35.7       -23.8       -23.9
P2_cons        -46.0       -53.4       -22.3       -35.9       -35.2       -29.6
P3_reeq          2.0       -36.0        -9.9       -41.3        -1.2        -9.8
P3_cons          5.9       -41.5       -22.8       -45.6        25.3       -18.8
P4_reeq        -68.1       -44.5       -16.2       -31.6       -40.3       -26.5
P4_cons        -86.2       -46.1       -16.8       -32.8       -46.6       -38.9
P5_reeq        -69.1       -55.8       -26.3       -33.3       -34.3       -32.2
P5_cons        -69.6       -56.2       -31.0       -34.8       -44.1  

In [26]:
episodes = pd.read_csv(SORTIE / "episodes_tension.csv")
en_tension = pd.Series(False, index=tout.index)
for e in episodes.itertuples():
    en_tension.loc[e.debut:e.creux] = True

print("tension %d seances | calme %d" % (en_tension.sum(), (~en_tension).sum()))


def rho_conditionnel(colonnes, masque):
    matrice = rendements.loc[masque.values, colonnes].corr(min_periods=100).to_numpy()
    haut = np.triu_indices_from(matrice, 1)
    paires = matrice[haut]
    return paires[~np.isnan(paires)].mean()


lignes = []
for pf in [f"P{i}" for i in range(1, 11)]:
    colonnes = [c for c in appartenance[appartenance.portefeuille == pf].titre
                if c in rendements.columns]
    en_crise = rho_conditionnel(colonnes, en_tension)
    au_calme = rho_conditionnel(colonnes, ~en_tension)
    n = len(colonnes)
    lignes.append({"portefeuille": pf, "n": n,
                   "rho_tension": en_crise, "rho_calme": au_calme,
                   "rapport": en_crise / au_calme,
                   "n_eff_calme": 1 / (au_calme + (1 - au_calme) / n),
                   "n_eff_tension": 1 / (en_crise + (1 - en_crise) / n)})

conditionnel = pd.DataFrame(lignes).set_index("portefeuille")
conditionnel.to_csv(SORTIE / "risque_correlation_conditionnelle.csv", encoding="utf-8")
print(conditionnel.round(4).to_string())

tension 1315 seances | calme 5394
                n  rho_tension  rho_calme  rapport  n_eff_calme  n_eff_tension
portefeuille                                                                  
P1            135       0.4326     0.3069   1.4099       3.2053         2.2891
P2            111       0.4625     0.3260   1.4188       3.0114         2.1396
P3             24       0.4638     0.3866   1.1996       2.4260         2.0570
P4             11       0.5641     0.3272   1.7238       2.5748         1.6565
P5             34       0.5246     0.3863   1.3580       2.4731         1.8567
P6             24       0.5614     0.5148   1.0905       1.8692         1.7252
P7             28       0.5475     0.4602   1.1898       2.0857         1.7741
P8              7       0.5313     0.4245   1.2515       1.9735         1.6715
P9             13       0.5837     0.4378   1.3335       2.0790         1.6240
P10            18       0.4797     0.3843   1.2485       2.3897         1.9660


In [27]:
marche = tout["SPY"].pct_change()

lignes = []
for serie in tout.columns:
    if serie == "SPY":
        continue
    d = pd.concat([tout[serie].pct_change().rename("s"), marche.rename("m")],
                  axis=1).dropna()
    hausse = d[d.m > 0]
    baisse = d[d.m < 0]

    lignes.append({
        "serie": serie,
        "beta": np.polyfit(d.m, d.s, 1)[0],
        "beta_hausse": np.polyfit(hausse.m, hausse.s, 1)[0],
        "beta_baisse": np.polyfit(baisse.m, baisse.s, 1)[0],
        "capture_hausse": hausse.s.mean() / hausse.m.mean(),
        "capture_baisse": baisse.s.mean() / baisse.m.mean(),
        "jours_hausse": len(hausse),
        "jours_baisse": len(baisse),
    })

conditionnel_marche = pd.DataFrame(lignes).set_index("serie")
conditionnel_marche["asymetrie"] = (conditionnel_marche.beta_baisse
                                    - conditionnel_marche.beta_hausse)
conditionnel_marche["gain_net"] = (conditionnel_marche.capture_hausse
                                   - conditionnel_marche.capture_baisse)

rang = [f"P{i}_{v}" for i in range(1, 11) for v in ("reeq", "cons")] \
       + ["RSP", "GSPC", "SP500TR", "SPXEW"]
conditionnel_marche = conditionnel_marche.reindex(
    [x for x in rang if x in conditionnel_marche.index])
conditionnel_marche.to_csv(SORTIE / "risque_beta_conditionnel.csv", encoding="utf-8")

print("jours de hausse du marche %d | de baisse %d"
      % (conditionnel_marche.jours_hausse.max(), conditionnel_marche.jours_baisse.max()))
print(conditionnel_marche[["beta", "beta_hausse", "beta_baisse", "asymetrie",
                           "capture_hausse", "capture_baisse", "gain_net"]]
      .round(3).to_string())

jours de hausse du marche 3654 | de baisse 3033
           beta  beta_hausse  beta_baisse  asymetrie  capture_hausse  capture_baisse  gain_net
serie                                                                                         
P1_reeq   1.050        1.007        1.025      0.017           1.138           1.051     0.087
P1_cons   1.099        1.046        1.077      0.032           1.193           1.106     0.087
P2_reeq   1.135        1.085        1.086      0.001           1.242           1.156     0.086
P2_cons   1.187        1.122        1.135      0.013           1.304           1.220     0.085
P3_reeq   0.724        0.729        0.799      0.069           0.717           0.633     0.083
P3_cons   0.779        0.789        0.888      0.098           0.752           0.658     0.094
P4_reeq   1.188        1.148        1.144     -0.004           1.324           1.165     0.159
P4_cons   1.271        1.189        1.213      0.025           1.402           1.322     0.080
P5

C:\Users\josue\AppData\Local\Temp\ipykernel_42416\1102194317.py:7: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  d = pd.concat([tout[serie].pct_change().rename("s"), marche.rename("m")],


In [28]:
NIVEAUX = [0.95, 0.99]

lignes = []
for serie in tout.columns:
    niveau = tout[serie].dropna()
    r = niveau.pct_change().dropna()
    mensuel = niveau.groupby(pd.Series(niveau.index).str[:7].values).last().pct_change().dropna()

    ligne = {"serie": serie, "obs": len(r)}
    for seuil in NIVEAUX:
        var = -r.quantile(1 - seuil)
        queue = r[r <= -var]
        ligne[f"var{int(seuil * 100)}"] = var
        ligne[f"perte_au_dela_{int(seuil * 100)}"] = -queue.mean()
        ligne[f"obs_queue_{int(seuil * 100)}"] = len(queue)

    ligne["var95_mensuelle"] = -mensuel.quantile(0.05)
    ligne["perte_au_dela_95_mensuelle"] = -mensuel[mensuel <= mensuel.quantile(0.05)].mean()
    lignes.append(ligne)

valeur_en_risque = pd.DataFrame(lignes).set_index("serie")
valeur_en_risque["aggravation_99"] = (valeur_en_risque.perte_au_dela_99
                                      / valeur_en_risque.var99)

rang = [f"P{i}_{v}" for i in range(1, 11) for v in ("reeq", "cons")] \
       + ["SPY", "RSP", "GSPC", "SP500TR", "SPXEW"]
valeur_en_risque = valeur_en_risque.reindex([x for x in rang if x in valeur_en_risque.index])
valeur_en_risque.to_csv(SORTIE / "risque_var.csv", encoding="utf-8")

print(valeur_en_risque.round(4).to_string())

           obs   var95  perte_au_dela_95  obs_queue_95   var99  perte_au_dela_99  obs_queue_99  var95_mensuelle  perte_au_dela_95_mensuelle  aggravation_99
serie                                                                                                                                                      
P1_reeq   6708  0.0210            0.0317           336  0.0358            0.0510            68           0.0812                      0.1115          1.4226
P1_cons   6708  0.0223            0.0350           336  0.0424            0.0575            68           0.0937                      0.1203          1.3571
P2_reeq   6708  0.0234            0.0346           336  0.0397            0.0537            68           0.0861                      0.1233          1.3541
P2_cons   6708  0.0252            0.0378           336  0.0454            0.0598            68           0.1008                      0.1283          1.3175
P3_reeq   6708  0.0168            0.0271           336  0.0311  

In [30]:
FENETRE_VAR = 252


def kupiec(n, x, p):
    """Rapport de vraisemblance de couverture, calcule en logarithmes.

    La forme directe (1-p)**(n-x) passe sous le plus petit flottant
    representable des que n depasse quelques centaines : 0,95**6093 vaut zero
    pour la machine, et son logarithme moins l'infini.
    """
    if x == 0 or x == n:
        return np.nan
    taux = x / n
    return -2 * ((n - x) * np.log(1 - p) + x * np.log(p)
                 - (n - x) * np.log(1 - taux) - x * np.log(taux))


lignes = []
for serie in tout.columns:
    r = tout[serie].dropna().pct_change().dropna()
    if len(r) < FENETRE_VAR + 250:
        continue

    for seuil in [0.95, 0.99]:
        var = r.rolling(FENETRE_VAR).quantile(1 - seuil).shift(1)
        d = pd.concat([r.rename("r"), var.rename("v")], axis=1).dropna()
        depasse = d.r < d.v
        n, x = len(d), int(depasse.sum())

        lignes.append({"serie": serie, "seuil": seuil, "obs": n,
                       "depassements": x, "attendus": round((1 - seuil) * n, 1),
                       "taux": x / n, "kupiec": kupiec(n, x, 1 - seuil),
                       "groupes": int((depasse & depasse.shift(1).fillna(False)).sum()),
                       "groupes_attendus": round(n * (1 - seuil) ** 2, 1)})

controle_var = pd.DataFrame(lignes)
controle_var["rejet"] = controle_var.kupiec > 3.84
controle_var.to_csv(SORTIE / "risque_depassements.csv", index=False, encoding="utf-8")

for seuil in [0.95, 0.99]:
    bloc = controle_var[controle_var.seuil == seuil].set_index("serie")
    print(f"=== VaR {seuil:.0%} | rejets {int(bloc.rejet.sum())} sur {len(bloc)}")
    print(bloc[["depassements", "attendus", "taux", "kupiec", "rejet",
                "groupes", "groupes_attendus"]].round(4).to_string())
    print()

C:\Users\josue\AppData\Local\Temp\ipykernel_42416\439416644.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "groupes": int((depasse & depasse.shift(1).fillna(False)).sum()),
C:\Users\josue\AppData\Local\Temp\ipykernel_42416\439416644.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "groupes": int((depasse & depasse.shift(1).fillna(False)).sum()),
C:\Users\josue\AppData\Local\Temp\ipykernel_42416\439416644.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call resul

=== VaR 95% | rejets 13 sur 25
          depassements  attendus    taux  kupiec  rejet  groupes  groupes_attendus
serie                                                                             
P1_reeq            363     322.8  0.0562  5.0746   True       50              16.1
P1_cons            373     322.8  0.0578  7.8429   True       47              16.1
P10_reeq           348     322.8  0.0539  2.0217  False       48              16.1
P10_cons           371     322.8  0.0575  7.2433   True       53              16.1
P2_reeq            353     322.8  0.0547  2.8901  False       45              16.1
P2_cons            366     322.8  0.0567  5.8445   True       42              16.1
P3_reeq            354     322.8  0.0548  3.0819  False       60              16.1
P3_cons            379     322.8  0.0587  9.7780   True       66              16.1
P4_reeq            341     322.8  0.0528  1.0614  False       27              16.1
P4_cons            353     322.8  0.0547  2.8901  False 

C:\Users\josue\AppData\Local\Temp\ipykernel_42416\439416644.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "groupes": int((depasse & depasse.shift(1).fillna(False)).sum()),
C:\Users\josue\AppData\Local\Temp\ipykernel_42416\439416644.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "groupes": int((depasse & depasse.shift(1).fillna(False)).sum()),
C:\Users\josue\AppData\Local\Temp\ipykernel_42416\439416644.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call resul

In [31]:
lignes = []
for serie in ordre:
    bloc = poids_mensuels[(poids_mensuels.serie == serie)
                          & (poids_mensuels.titre != "_tresorerie")]
    for date, groupe in bloc.groupby("date"):
        w = groupe.poids
        w = w[w > 0]
        w = w / w.sum()
        lignes.append({"serie": serie, "date": date, "titres": len(w),
                       "poids_max": w.max(), "premier": groupe.set_index("titre").poids.idxmax(),
                       "hhi": float((w ** 2).sum()), "n_eff": 1 / float((w ** 2).sum()),
                       "part_5": w.nlargest(5).sum()})

evolution = pd.DataFrame(lignes)
evolution.to_csv(SORTIE / "risque_concentration_evolution.csv", index=False, encoding="utf-8")

resume = evolution.groupby("serie").agg(
    poids_max_min=("poids_max", "min"), poids_max_fin=("poids_max", "last"),
    poids_max_max=("poids_max", "max"),
    n_eff_debut=("n_eff", "first"), n_eff_min=("n_eff", "min"), n_eff_fin=("n_eff", "last"),
    part_5_fin=("part_5", "last"))
print(resume.reindex(ordre).round(4).to_string())

for serie in ["P1_cons", "P1_reeq"]:
    annuel = evolution[evolution.serie == serie].copy()
    annuel["an"] = annuel.date.str[:4]
    annuel = annuel.groupby("an").last()
    print(f"\n=== {serie}, releve de fin d'annee")
    print(annuel[["titres", "premier", "poids_max", "n_eff", "part_5"]]
          .round(4).to_string())

          poids_max_min  poids_max_fin  poids_max_max  n_eff_debut  n_eff_min  n_eff_fin  part_5_fin
serie                                                                                               
P1_cons          0.0185         0.1944         0.2501      88.3290    11.2799    11.5687      0.5215
P1_reeq          0.0090         0.0368         0.0723      88.3290    67.1424   106.8626      0.1112
P2_cons          0.0242         0.2147         0.2879      67.5132     9.5714     9.8776      0.5440
P2_reeq          0.0110         0.0431         0.0788      67.5132    52.1073    86.1904      0.1303
P3_cons          0.0558         0.5643         0.6718      20.8161     2.1646     2.9904      0.7492
P3_reeq          0.0444         0.0531         0.2841      20.8161     9.4570    23.6670      0.2449
P4_cons          0.2045         0.3843         0.6208       3.5041     2.2910     4.1939      0.8613
P4_reeq          0.1066         0.1228         0.6257       3.5041     2.2910    10.3391   

In [32]:
def retrait(serie, critere="risque", fenetre=252):
    w = poids_mensuels[(poids_mensuels.serie == serie)
                       & (poids_mensuels.date == derniere)
                       & (poids_mensuels.titre != "_tresorerie")].set_index("titre").poids
    w = w[w > 0]
    colonnes = [c for c in w.index if c in rendements.columns]
    w = w[colonnes] / w[colonnes].sum()

    covariance = rendements[colonnes].iloc[-fenetre:].cov() * 252
    matrice = covariance.to_numpy()

    def volatilite(poids):
        return float(np.sqrt(poids @ matrice @ poids))

    depart = volatilite(w.to_numpy())

    if critere == "poids":
        vise = w.idxmax()
    else:
        part = w.to_numpy() * (matrice @ w.to_numpy())
        vise = colonnes[int(np.argmax(part))]

    garde = [c for c in colonnes if c != vise]
    indices = [colonnes.index(c) for c in garde]
    reduit = w[garde].to_numpy()
    reduit = reduit / reduit.sum()
    apres = float(np.sqrt(reduit @ matrice[np.ix_(indices, indices)] @ reduit))

    return {"serie": serie, "retire": vise, "poids_retire": float(w[vise]),
            "vol_avant": depart, "vol_apres": apres,
            "baisse_pt": 100 * (depart - apres), "baisse_pct": 100 * (apres / depart - 1)}


marginal = pd.DataFrame([retrait(s) for s in ordre]).set_index("serie")
marginal.to_csv(SORTIE / "risque_marginal.csv", encoding="utf-8")
print(marginal.round(4).to_string())

         retire  poids_retire  vol_avant  vol_apres  baisse_pt  baisse_pct
serie                                                                     
P1_cons    SNDK        0.1938     0.4099     0.2830    12.6840    -30.9477
P1_reeq    SNDK        0.0368     0.2493     0.2266     2.2667     -9.0935
P2_cons    SNDK        0.2147     0.4412     0.3031    13.8091    -31.3007
P2_reeq    SNDK        0.0431     0.2856     0.2601     2.5500     -8.9295
P3_cons     TPL        0.5643     0.2968     0.1675    12.9289    -43.5615
P3_reeq     TPL        0.0483     0.1253     0.1229     0.2467     -1.9688
P4_cons     APP        0.2648     0.3523     0.3143     3.8050    -10.7996
P4_reeq    ORCL        0.0873     0.2297     0.2187     1.0964     -4.7741
P5_cons    SNDK        0.4314     0.6466     0.3632    28.3409    -43.8325
P5_reeq    SNDK        0.1045     0.4531     0.4014     5.1707    -11.4108
P6_cons     VST        0.1535     0.2085     0.1777     3.0868    -14.8018
P6_reeq     VST        0.

In [33]:
CHOC = -0.50


def choc(serie, fenetre=252):
    w = poids_mensuels[(poids_mensuels.serie == serie)
                       & (poids_mensuels.date == derniere)
                       & (poids_mensuels.titre != "_tresorerie")].set_index("titre").poids
    w = w[w > 0]
    colonnes = [c for c in w.index if c in rendements.columns]
    w = w[colonnes] / w[colonnes].sum()

    matrice = (rendements[colonnes].iloc[-fenetre:].cov() * 252).to_numpy()
    poids = w.to_numpy()

    part = poids * (matrice @ poids)
    j = int(np.argmax(part))
    vise = colonnes[j]

    beta = matrice[:, j] / matrice[j, j]

    isole = float(poids[j]) * CHOC
    propage = float(poids @ beta) * CHOC

    return {"serie": serie, "titre_choque": vise, "poids": float(poids[j]),
            "perte_isolee": isole, "perte_propagee": propage,
            "beta_moyen": float(poids @ beta),
            "amplification": propage / isole}


chocs = pd.DataFrame([choc(s) for s in ordre]).set_index("serie")
chocs.to_csv(SORTIE / "risque_choc.csv", encoding="utf-8")
print(f"choc applique : {CHOC:.0%} sur le premier contributeur au risque")
print(chocs.round(4).to_string())

choc applique : -50% sur le premier contributeur au risque
         titre_choque   poids  perte_isolee  perte_propagee  beta_moyen  amplification
serie                                                                                 
P1_cons          SNDK  0.1938       -0.0969         -0.1585      0.3170         1.6358
P1_reeq          SNDK  0.0368       -0.0184         -0.0817      0.1634         4.4399
P2_cons          SNDK  0.2147       -0.1074         -0.1719      0.3438         1.6011
P2_reeq          SNDK  0.0431       -0.0216         -0.0945      0.1890         4.3837
P3_cons           TPL  0.5643       -0.2821         -0.2991      0.5982         1.0600
P3_reeq           TPL  0.0483       -0.0241         -0.0575      0.1149         2.3805
P4_cons           APP  0.2648       -0.1324         -0.1881      0.3762         1.4207
P4_reeq          ORCL  0.0873       -0.0436         -0.1025      0.2051         2.3503
P5_cons          SNDK  0.4314       -0.2157         -0.2679      0.5359

In [34]:
mesures = []
for pf in [f"P{i}" for i in range(1, 11)]:
    ligne = {"portefeuille": pf}
    for version in ["reeq", "cons"]:
        serie = f"{pf}_{version}"
        niveau = valeurs[serie].dropna()
        r = niveau.pct_change().dropna()
        excedent = (r - sans_risque.reindex(r.index)).dropna()
        baisse = np.sqrt((np.minimum(excedent, 0) ** 2).mean()) * np.sqrt(252)

        ligne[f"vol_{version}"] = r.std() * np.sqrt(252)
        ligne[f"semivol_{version}"] = baisse
        ligne[f"repli_{version}"] = (niveau / niveau.cummax() - 1).min()
        ligne[f"var99_{version}"] = -r.quantile(0.01)
        ligne[f"es99_{version}"] = -r[r <= r.quantile(0.01)].mean()
        ligne[f"sharpe_{version}"] = excedent.mean() * np.sqrt(252) / r.std()
        ligne[f"sortino_{version}"] = excedent.mean() * 252 / baisse
        ligne[f"annualise_{version}"] = (niveau.iloc[-1] / niveau.iloc[0]) ** (252 / len(niveau)) - 1
    mesures.append(ligne)

arbitrage = pd.DataFrame(mesures).set_index("portefeuille")

for critere, sens in [("vol", "bas"), ("semivol", "bas"), ("repli", "haut"),
                      ("var99", "bas"), ("es99", "bas"),
                      ("sharpe", "haut"), ("sortino", "haut"), ("annualise", "haut")]:
    a, b = arbitrage[f"{critere}_reeq"], arbitrage[f"{critere}_cons"]
    gagne = (a < b) if sens == "bas" else (a > b)
    arbitrage[f"gagnant_{critere}"] = np.where(gagne, "reeq", "cons")

colonnes = [c for c in arbitrage.columns if c.startswith("gagnant_")]
compte = arbitrage[colonnes].apply(lambda c: (c == "reeq").sum())

arbitrage.to_csv(SORTIE / "risque_arbitrage_gestion.csv", encoding="utf-8")
print(arbitrage[colonnes].to_string())
print()
print("victoires du reequilibrage, sur 10 portefeuilles :")
print(compte.to_string())

             gagnant_vol gagnant_semivol gagnant_repli gagnant_var99 gagnant_es99 gagnant_sharpe gagnant_sortino gagnant_annualise
portefeuille                                                                                                                      
P1                  reeq            reeq          reeq          reeq         reeq           reeq            reeq              cons
P2                  reeq            reeq          reeq          reeq         reeq           reeq            reeq              reeq
P3                  reeq            reeq          reeq          reeq         reeq           reeq            reeq              cons
P4                  reeq            reeq          reeq          reeq         reeq           reeq            reeq              reeq
P5                  reeq            reeq          reeq          reeq         reeq           cons            cons              cons
P6                  reeq            reeq          cons          reeq         reeq  

In [35]:
def mensualiser(niveau):
    return niveau.groupby(pd.Series(niveau.index).str[:7].values).last()


def comparer(frequence):
    mesures = []
    for pf in [f"P{i}" for i in range(1, 11)]:
        ligne = {"portefeuille": pf}
        for version in ["reeq", "cons"]:
            niveau = valeurs[f"{pf}_{version}"].dropna()
            if frequence == "mensuelle":
                niveau = mensualiser(niveau)
                racine, periodes = np.sqrt(12), 12
            else:
                racine, periodes = np.sqrt(252), 252

            r = niveau.pct_change().dropna()
            baisse = np.sqrt((np.minimum(r, 0) ** 2).mean()) * racine

            ligne[f"vol_{version}"] = r.std() * racine
            ligne[f"semivol_{version}"] = baisse
            ligne[f"repli_{version}"] = (niveau / niveau.cummax() - 1).min()
            ligne[f"var95_{version}"] = -r.quantile(0.05)
            ligne[f"es95_{version}"] = -r[r <= r.quantile(0.05)].mean()
            ligne[f"annualise_{version}"] = (niveau.iloc[-1] / niveau.iloc[0]) ** (
                periodes / len(niveau)) - 1
        mesures.append(ligne)

    table = pd.DataFrame(mesures).set_index("portefeuille")
    for critere, sens in [("vol", "bas"), ("semivol", "bas"), ("repli", "haut"),
                          ("var95", "bas"), ("es95", "bas"), ("annualise", "haut")]:
        a, b = table[f"{critere}_reeq"], table[f"{critere}_cons"]
        table[f"g_{critere}"] = np.where((a < b) if sens == "bas" else (a > b), "reeq", "cons")
    return table


quotidien = comparer("quotidienne")
mensuel = comparer("mensuelle")

colonnes = [c for c in quotidien.columns if c.startswith("g_")]
resultat = pd.DataFrame({"quotidien": (quotidien[colonnes] == "reeq").sum(),
                         "mensuel": (mensuel[colonnes] == "reeq").sum()})
resultat["accord"] = [(quotidien[c] == mensuel[c]).sum() for c in colonnes]

print("victoires du reequilibrage sur dix portefeuilles, et accord entre frequences")
print(resultat.to_string())
print()
print("desaccords, portefeuille par portefeuille")
for c in colonnes:
    d = quotidien.index[quotidien[c] != mensuel[c]].tolist()
    if d:
        print(f"  {c[2:]:12s} {d}")

victoires du reequilibrage sur dix portefeuilles, et accord entre frequences
             quotidien  mensuel  accord
g_vol                9        9      10
g_semivol            9        9      10
g_repli              7        8       9
g_var95              8        8       8
g_es95               9        8       9
g_annualise          6        6      10

desaccords, portefeuille par portefeuille
  repli        ['P6']
  var95        ['P6', 'P9']
  es95         ['P6']


In [36]:
FENETRE_LONGUE = 756

lignes = []
for pf in [f"P{i}" for i in range(1, 11)]:
    a = valeurs[f"{pf}_reeq"].pct_change()
    b = valeurs[f"{pf}_cons"].pct_change()

    vol_a = a.rolling(FENETRE_LONGUE).std()
    vol_b = b.rolling(FENETRE_LONGUE).std()
    d = pd.concat([vol_a.rename("reeq"), vol_b.rename("cons")], axis=1).dropna()
    d = d.iloc[::21]

    gagne = d.reeq < d.cons
    ecart = (d.reeq / d.cons - 1)

    lignes.append({"portefeuille": pf, "fenetres": len(d),
                   "part_reeq_gagne": gagne.mean(),
                   "ecart_median": ecart.median(),
                   "ecart_min": ecart.min(), "ecart_max": ecart.max(),
                   "premiere_perte": d.index[~gagne][0] if (~gagne).any() else None,
                   "derniere_perte": d.index[~gagne][-1] if (~gagne).any() else None})

stabilite = pd.DataFrame(lignes).set_index("portefeuille")
stabilite.to_csv(SORTIE / "risque_stabilite_fenetre.csv", encoding="utf-8")
print(f"fenetres glissantes de {FENETRE_LONGUE} seances, relevees tous les 21 jours")
print(stabilite.round(4).to_string())

fenetres glissantes de 756 seances, relevees tous les 21 jours
              fenetres  part_reeq_gagne  ecart_median  ecart_min  ecart_max premiere_perte derniere_perte
portefeuille                                                                                             
P1                 284           0.6866       -0.0125    -0.3246     0.1768     2003-01-08     2018-01-12
P2                 284           0.7852       -0.0193    -0.2880     0.1517     2003-01-08     2012-03-12
P3                 284           0.7148       -0.0915    -0.5302     0.1163     2003-01-08     2017-02-13
P4                 284           0.6866       -0.0448    -0.3335     0.3939     2003-05-09     2014-04-14
P5                 284           0.8028       -0.0354    -0.2933     0.0513     2003-01-08     2015-03-16
P6                 284           0.3944        0.0089    -0.2653     0.0869     2003-02-07     2021-12-14
P7                 284           0.5986       -0.0104    -0.1898     0.0742     2003-01-0

In [37]:
DEBUT_P4 = "2004-08-19"


def statistiques(depuis=None):
    lignes = []
    for pf in [f"P{i}" for i in range(1, 11)]:
        ligne = {"portefeuille": pf}
        for version in ["reeq", "cons"]:
            niveau = valeurs[f"{pf}_{version}"].dropna()
            if depuis:
                niveau = niveau.loc[depuis:]
            r = niveau.pct_change().dropna()
            excedent = (r - sans_risque.reindex(r.index)).dropna()

            ligne[f"vol_{version}"] = r.std() * np.sqrt(252)
            ligne[f"repli_{version}"] = (niveau / niveau.cummax() - 1).min()
            ligne[f"sharpe_{version}"] = excedent.mean() * np.sqrt(252) / r.std()
            ligne[f"annualise_{version}"] = (niveau.iloc[-1] / niveau.iloc[0]) ** (
                252 / len(niveau)) - 1
        lignes.append(ligne)

    table = pd.DataFrame(lignes).set_index("portefeuille")
    for critere, sens in [("vol", "bas"), ("repli", "haut"),
                          ("sharpe", "haut"), ("annualise", "haut")]:
        a, b = table[f"{critere}_reeq"], table[f"{critere}_cons"]
        table[f"g_{critere}"] = np.where((a < b) if sens == "bas" else (a > b), "reeq", "cons")
    return table


complete = statistiques()
tronquee = statistiques(DEBUT_P4)

colonnes = [c for c in complete.columns if c.startswith("g_")]
print("verdicts sur la periode complete")
print(complete[colonnes].to_string())
print()
print(f"verdicts depuis le {DEBUT_P4}")
print(tronquee[colonnes].to_string())
print()
print("desaccords entre les deux lectures")
for c in colonnes:
    d = complete.index[complete[c] != tronquee[c]].tolist()
    print(f"  {c[2:]:12s} {d if d else 'aucun'}")
print()
print("P4 en detail")
print(pd.DataFrame({"complete": complete.loc["P4"], "depuis_2004": tronquee.loc["P4"]}).to_string())

verdicts sur la periode complete
             g_vol g_repli g_sharpe g_annualise
portefeuille                                   
P1            reeq    reeq     reeq        cons
P2            reeq    reeq     reeq        reeq
P3            reeq    reeq     reeq        cons
P4            reeq    reeq     reeq        reeq
P5            reeq    reeq     cons        cons
P6            reeq    cons     reeq        reeq
P7            reeq    cons     reeq        reeq
P8            cons    cons     reeq        reeq
P9            reeq    reeq     reeq        reeq
P10           reeq    reeq     reeq        cons

verdicts depuis le 2004-08-19
             g_vol g_repli g_sharpe g_annualise
portefeuille                                   
P1            reeq    reeq     reeq        cons
P2            reeq    reeq     reeq        cons
P3            reeq    reeq     reeq        cons
P4            reeq    reeq     reeq        reeq
P5            reeq    cons     cons        cons
P6            reeq    co

In [38]:
variations = pd.read_csv(RACINE / "data" / "review" / "variations_finition_2026-09-08.csv",
                         encoding="utf-8", engine="python")
suspectes = variations[variations.statut.str.startswith("non_corrobor")
                       & variations.dans_historique_admissible]
jours_suspects = set(suspectes.date)

print(f"{len(variations)} variations examinees | {len(suspectes)} non corroborees")
print(f"{suspectes.titre.nunique()} titres | de {suspectes.date.min()} a {suspectes.date.max()}")

lignes = []
for pf in [f"P{i}" for i in range(1, 11)]:
    ligne = {"portefeuille": pf}
    for version in ["reeq", "cons"]:
        r = valeurs[f"{pf}_{version}"].pct_change().dropna()
        garde = ~r.index.isin(jours_suspects)

        for suffixe, serie in [("", r), ("_hors", r[garde])]:
            excedent = (serie - sans_risque.reindex(serie.index)).dropna()
            ligne[f"vol{suffixe}_{version}"] = serie.std() * np.sqrt(252)
            ligne[f"sharpe{suffixe}_{version}"] = excedent.mean() * np.sqrt(252) / serie.std()
        ligne[f"jours_retires_{version}"] = int((~garde).sum())
    lignes.append(ligne)

sensibilite = pd.DataFrame(lignes).set_index("portefeuille")
for critere, sens in [("vol", "bas"), ("sharpe", "haut")]:
    for suffixe in ["", "_hors"]:
        a = sensibilite[f"{critere}{suffixe}_reeq"]
        b = sensibilite[f"{critere}{suffixe}_cons"]
        sensibilite[f"g_{critere}{suffixe}"] = np.where(
            (a < b) if sens == "bas" else (a > b), "reeq", "cons")

sensibilite.to_csv(SORTIE / "risque_sensibilite_extremes.csv", encoding="utf-8")
print(sensibilite[["g_vol", "g_vol_hors", "g_sharpe", "g_sharpe_hors"]].to_string())
print("verdicts inchanges : volatilite %d/10, Sharpe %d/10"
      % ((sensibilite.g_vol == sensibilite.g_vol_hors).sum(),
         (sensibilite.g_sharpe == sensibilite.g_sharpe_hors).sum()))
print(sensibilite[["vol_reeq", "vol_hors_reeq", "vol_cons", "vol_hors_cons"]].round(4).to_string())

111 variations examinees | 82 non corroborees
28 titres | de 2000-01-19 a 2016-04-22
             g_vol g_vol_hors g_sharpe g_sharpe_hors
portefeuille                                        
P1            reeq       reeq     reeq          reeq
P2            reeq       reeq     reeq          reeq
P3            reeq       reeq     reeq          reeq
P4            reeq       reeq     reeq          reeq
P5            reeq       reeq     cons          cons
P6            reeq       reeq     reeq          reeq
P7            reeq       reeq     reeq          reeq
P8            cons       cons     reeq          reeq
P9            reeq       reeq     reeq          reeq
P10           reeq       reeq     reeq          cons
verdicts inchanges : volatilite 10/10, Sharpe 9/10
              vol_reeq  vol_hors_reeq  vol_cons  vol_hors_cons
portefeuille                                                  
P1              0.2173         0.2147    0.2352         0.2333
P2              0.2382         0.2348  